### Note: ROC/AUC doesn't work on multi-class classification and only works with Binary Classification

In [44]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report,f1_score, accuracy_score, confusion_matrix,log_loss
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
import os
os.chdir('/home/pgcp-ai/MachineLearning/Cases/Image_Segmentation/')

In [2]:
image = pd.read_csv("Image_Segmentation.csv")

In [3]:
image

,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [4]:
image.isna().sum().sum()

0

In [5]:
le = LabelEncoder()

In [6]:
ss = StandardScaler().set_output(transform='pandas')

In [7]:
image['Class'] = le.fit_transform(image['Class'])

In [8]:
X, y = image.drop('Class', axis = 1), image['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26,stratify=image['Class'])
y_train

94     1
36     5
16     0
91     1
108    1
      ..
38     5
136    6
85     2
148    6
61     2
Name: Class, Length: 146, dtype: int64

In [9]:
X_train_trf = ss.fit_transform(X_train)
X_test_trf = ss.transform(X_test)

In [10]:
scores=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in k:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train)
    y_pred = knn.predict(X_test)
    y_pred_prob = knn.predict_proba(X_test)
    score = log_loss(y_test,y_pred_prob)
    scores.append([i,score])

df_scores = pd.DataFrame(scores,columns=['k','scores'])
df_scores.sort_values('scores',ascending=True)

,k,scores
5,6,0.971258
6,7,0.981489
7,8,1.016964
8,9,1.049631
9,10,1.084273
4,5,1.506703
2,3,1.996419
3,4,2.015082
1,2,3.036645
0,1,4.004850


In [11]:
scores_scaled=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in k:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_trf,y_train)
    y_pred = knn.predict(X_test_trf)
    y_pred_prob = knn.predict_proba(X_test_trf)
    score = log_loss(y_test,y_pred_prob)
    scores.append([i,score])

df_scores_scaled = pd.DataFrame(scores,columns=['k','scores'])
df_scores_scaled.sort_values('scores',ascending=True)

,k,scores
17,8,0.352821
18,9,0.381166
19,10,0.399369
15,6,0.815084
16,7,0.856190
5,6,0.971258
6,7,0.981489
7,8,1.016964
8,9,1.049631
9,10,1.084273


In [13]:
bm_non_scaled = KNeighborsClassifier(n_neighbors=6)
bm_non_scaled.fit(X_train,y_train)
y_pred_non_scaled = bm_non_scaled.predict(X_test)
y_pred_prob_non_scaled = bm_non_scaled.predict_proba(X_test)
acc_non_scaled = accuracy_score(y_test,y_pred)
y_pred_labels_non_scaled = le.inverse_transform(y_pred)
y_pred_labels_non_scaled

array(['SKY', 'BRICKFACE', 'CEMENT', 'GRASS', 'PATH', 'FOLIAGE',
       'BRICKFACE', 'BRICKFACE', 'WINDOW', 'GRASS', 'BRICKFACE',
       'FOLIAGE', 'CEMENT', 'CEMENT', 'WINDOW', 'GRASS', 'SKY',
       'BRICKFACE', 'BRICKFACE', 'BRICKFACE', 'GRASS', 'PATH', 'WINDOW',
       'PATH', 'PATH', 'SKY', 'GRASS', 'PATH', 'GRASS', 'CEMENT',
       'CEMENT', 'GRASS', 'BRICKFACE', 'BRICKFACE', 'SKY', 'PATH', 'PATH',
       'GRASS', 'FOLIAGE', 'SKY', 'CEMENT', 'BRICKFACE', 'FOLIAGE',
       'PATH', 'PATH', 'PATH', 'SKY', 'CEMENT', 'GRASS', 'BRICKFACE',
       'WINDOW', 'SKY', 'BRICKFACE', 'PATH', 'SKY', 'FOLIAGE', 'FOLIAGE',
       'FOLIAGE', 'BRICKFACE', 'BRICKFACE', 'BRICKFACE', 'FOLIAGE', 'SKY'],
      dtype=object)

In [14]:
print(classification_report(y_test,y_pred_non_scaled))

              precision    recall  f1-score   support

           0       0.57      0.89      0.70         9
           1       1.00      0.89      0.94         9
           2       1.00      0.78      0.88         9
           3       0.90      1.00      0.95         9
           4       1.00      0.89      0.94         9
           5       1.00      1.00      1.00         9
           6       0.86      0.67      0.75         9

    accuracy                           0.87        63
   macro avg       0.90      0.87      0.88        63
weighted avg       0.90      0.87      0.88        63



In [34]:
tst = pd.read_csv("tst_img.csv")
tst

,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,22,90,10,0,0,0.666668,0.044444,0.880000,0.562963,112.000000,105.888885,128.555560,106.000000,-22.777779,45.222220,-22.444445,128.555560,0.179697,-2.097815
1,210,200,9,0,0,1.300000,0.998145,1.611111,1.123816,49.481480,45.000000,60.666668,43.000000,-14.111111,35.000000,-19.444445,60.666668,0.290788,-1.987599
2,240,184,9,0,0,0.500000,0.077778,0.777778,0.785185,11.851851,9.777778,9.888889,15.888889,-5.000000,-5.888889,13.000000,15.888889,0.500000,2.128646
3,130,191,9,0,0,1.000000,0.400000,1.500000,1.011111,7.333334,5.333334,5.000000,11.222222,-7.000000,-5.666666,11.666667,11.222222,0.535820,2.122422


In [35]:
image.columns

Index(['Class', 'region.centroid.col', 'region.centroid.row',
       'region.pixel.count', 'short.line.density.5', 'short.line.density.2',
       'vedge.mean', 'vegde.sd', 'hedge.mean', 'hedge.sd', 'intensity.mean',
       'rawred.mean', 'rawblue.mean', 'rawgreen.mean', 'exred.mean',
       'exblue.mean', 'exgreen.mean', 'value.mean', 'saturation.mean',
       'hue-mean'],
      dtype='object')

In [36]:
X_train.columns

Index(['region.centroid.col', 'region.centroid.row', 'region.pixel.count',
       'short.line.density.5', 'short.line.density.2', 'vedge.mean',
       'vegde.sd', 'hedge.mean', 'hedge.sd', 'intensity.mean', 'rawred.mean',
       'rawblue.mean', 'rawgreen.mean', 'exred.mean', 'exblue.mean',
       'exgreen.mean', 'value.mean', 'saturation.mean', 'hue-mean'],
      dtype='object')

In [37]:
tst.columns

Index(['region.centroid.col', 'region.centroid.row', 'region.pixel.count',
       'short.line.density.5', 'short.line.density.2', 'vedge.mean',
       'vegde.sd', 'hedge.mean', 'hedge.sd', 'intensity.mean', 'rawred.mean',
       'rawblue.mean', 'rawgreen.mean', 'exred.mean', 'exblue.mean',
       'exgreen.mean', 'value.mean', 'saturation.mean', 'hue-mean'],
      dtype='object')

In [38]:
bm_non_scaled = KNeighborsClassifier(n_neighbors=6)
bm_non_scaled.fit(X,y)

KNeighborsClassifier(n_neighbors=6)

In [39]:
tst['class_label_non_scaled'] = le.inverse_transform(bm_non_scaled.predict(tst))

In [40]:
tst

,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean,class_label_non_scaled
0,22,90,10,0,0,0.666668,0.044444,0.880000,0.562963,112.000000,105.888885,128.555560,106.000000,-22.777779,45.222220,-22.444445,128.555560,0.179697,-2.097815,SKY
1,210,200,9,0,0,1.300000,0.998145,1.611111,1.123816,49.481480,45.000000,60.666668,43.000000,-14.111111,35.000000,-19.444445,60.666668,0.290788,-1.987599,PATH
2,240,184,9,0,0,0.500000,0.077778,0.777778,0.785185,11.851851,9.777778,9.888889,15.888889,-5.000000,-5.888889,13.000000,15.888889,0.500000,2.128646,GRASS
3,130,191,9,0,0,1.000000,0.400000,1.500000,1.011111,7.333334,5.333334,5.000000,11.222222,-7.000000,-5.666666,11.666667,11.222222,0.535820,2.122422,GRASS


In [41]:
bm_scaled = KNeighborsClassifier(n_neighbors=8)
X_scaled = ss.fit_transform(X)
tst_scaled = tst.drop('class_label_non_scaled',axis=1)
tst_scaled = ss.transform(tst_scaled)
bm_scaled.fit(X_scaled,y)
tst['class_label_scaled']=le.inverse_transform(bm_scaled.predict(tst_scaled))

In [42]:
tst

,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,...,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean,class_label_non_scaled,class_label_scaled
0,22,90,10,0,0,0.666668,0.044444,0.880000,0.562963,112.000000,...,128.555560,106.000000,-22.777779,45.222220,-22.444445,128.555560,0.179697,-2.097815,SKY,SKY
1,210,200,9,0,0,1.300000,0.998145,1.611111,1.123816,49.481480,...,60.666668,43.000000,-14.111111,35.000000,-19.444445,60.666668,0.290788,-1.987599,PATH,PATH
2,240,184,9,0,0,0.500000,0.077778,0.777778,0.785185,11.851851,...,9.888889,15.888889,-5.000000,-5.888889,13.000000,15.888889,0.500000,2.128646,GRASS,GRASS
3,130,191,9,0,0,1.000000,0.400000,1.500000,1.011111,7.333334,...,5.000000,11.222222,-7.000000,-5.666666,11.666667,11.222222,0.535820,2.122422,GRASS,GRASS


In [43]:
image = pd.read_csv("Image_Segmentation.csv")

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=26, stratify = image['Class'])

In [48]:
minmax = MinMaxScaler()
X_train_trans = minmax.fit_transform(X_train)
X_test_trans = minmax.transform(X_test)

In [49]:
scores_scaled=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in k:
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_trans,y_train)
    y_pred = knn.predict(X_test_trans)
    y_pred_prob = knn.predict_proba(X_test_trans)
    score = log_loss(y_test,y_pred_prob)
    scores.append([i,score])

df_scores_scaled = pd.DataFrame(scores,columns=['k','scores'])
df_scores_scaled.sort_values('scores',ascending=True)

,k,scores
22,3,0.165250
23,4,0.197000
24,5,0.256929
25,6,0.281758
26,7,0.304352
27,8,0.343368
17,8,0.352821
18,9,0.381166
28,9,0.385612
19,10,0.399369


#### Concrete Strength

In [52]:
os.chdir('/home/pgcp-ai/MachineLearning/Cases/Concrete_Strength/')
conc = pd.read_csv("Concrete_Data.csv")
X,y = conc.drop("Strength",axis=1) , conc["Strength"]